<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/Notebook_final_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación y Carga del Ecosisitema Base

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python Pillow numpy google-generativeai pydantic huggingface_hub

import torch
import cv2
import json
import requests
import numpy as np
import urllib.parse
from PIL import Image, ImageDraw
from io import BytesIO
from pydantic import BaseModel, Field
from typing import Tuple, Dict, Any, List

from google import genai
from google.genai import types
from google.colab import userdata
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, DDIMScheduler
from diffusers.utils import load_image, make_image_grid
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

print("🚀 Iniciando Motor DecoAI v2.0 (Materiales y Multi-tienda)...")

# --- CREDENCIALES ---
GOOGLE_API_KEY = userdata.get('clave_API_gemini')
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# --- MODELOS ---
print("🧠 Cargando IA de Segmentación Universal...")
processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-tiny-coco-panoptic")
segmentation_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-tiny-coco-panoptic",
    torch_dtype=torch.float16
).to("cuda")

print("📐 Cargando IA Estructural (ControlNet MLSD)...")
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_mlsd", torch_dtype=torch.float16)

print("🎨 Cargando IA Generativa (Inpainting+ControlNet)...")
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

print("✅ Sistemas en línea y listos.")

El Cerebro (Pipeline completo y para toda la casa)

In [ ]:
# --- ESQUEMAS DE DATOS UNIVERSALES ---
class ProductoRecomendado(BaseModel):
    nombre: str = Field(description="Nombre del producto (ej: Suelo porcelánico, Grifo negro mate, Sofá cama, Horno empotrable)")
    categoria: str = Field(description="'mueble', 'iluminacion', 'suelo', 'pared', 'electrodomestico', 'sanitario' o 'decoracion'")
    tienda_sugerida: str = Field(description="Ej: Leroy Merlin, IKEA, Amazon, Zara Home, Bauhaus, Maisons du Monde")
    precio_estimado_eur: int
    detalles_medidas: str = Field(description="Ej: 150x190cm, 5 litros (pintura), 15m2 (suelo), o 'Estándar'")

class AnalisisDecoracionPro(BaseModel):
    tipo_estancia: str = Field(description="Identifica la foto: 'Baño', 'Cocina', 'Salón', 'Dormitorio', 'Comedor', 'Terraza', etc.")
    modo_generacion: str = Field(description="'completo' (cambia toda la estancia) o 'parcial' (solo un elemento/superficie)")
    objeto_a_reemplazar: str = Field(description="Si es parcial, qué quitar (ej: 'suelo', 'pared', 'nevera', 'inodoro'). Si es completo: 'todo'.")
    analisis_diseno: str = Field(description="Breve justificación del nuevo diseño adaptado al tipo de estancia.")
    prompt_generacion: str = Field(description="Prompt en inglés para Stable Diffusion detallando texturas, luz y elementos arquitectónicos.")
    presupuesto_total_estimado: int
    lista_compra: List[ProductoRecomendado]

# --- FUNCIONES DE SOPORTE ---
def preparar_imagen_pro(ruta_o_url: str, max_size: int = 512) -> Image.Image:
    try:
        if ruta_o_url.startswith('http'):
            # Disfrazamos la petición para que las webs no nos bloqueen por ser un bot
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            response = requests.get(ruta_o_url, headers=headers, timeout=10)
            response.raise_for_status() # Lanza error si la URL da 404 o 403
            img = Image.open(BytesIO(response.content)).convert("RGB")
        else:
            img = Image.open(ruta_o_url).convert("RGB")
            
        ancho, alto = img.size
        ratio = alto / ancho
        nuevo_ancho, nuevo_alto = (max_size, int(max_size * ratio)) if ancho > alto else (int(max_size / ratio), max_size)
        nuevo_ancho, nuevo_alto = (nuevo_ancho // 8) * 8, (nuevo_alto // 8) * 8
        return img.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)
    except Exception as e:
        raise ValueError(f"Error descargando la imagen. Prueba con otra URL o sube el archivo directamente. Detalle: {e}")

def generar_mascara(imagen: Image.Image, nombre_objeto: str) -> Image.Image:
    if nombre_objeto.lower() == 'todo':
        return Image.new("L", imagen.size, 255)

    print(f"   🔍 Buscando '{nombre_objeto}' en la estancia...")
    inputs = processor(images=imagen, return_tensors="pt").to("cuda")
    
    # Convertimos los píxeles a float16 para evitar errores de memoria
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
        
    with torch.no_grad():
        outputs = segmentation_model(**inputs)
    
    segmentation_map = processor.post_process_panoptic_segmentation(outputs, target_sizes=[imagen.size[::-1]])[0]
    mapa = segmentation_map['segmentation'].cpu().numpy()
    info = segmentation_map['segments_info']
    
    mascara = np.zeros_like(mapa, dtype=np.uint8)
    
    # --- DICCIONARIO AMPLIADO (TODA LA CASA) ---
    target_ids = []
    obj = nombre_objeto.lower()
    
    # Superficies
    if 'suelo' in obj or 'floor' in obj: target_ids = [54, 55] 
    elif 'pared' in obj or 'wall' in obj or 'azulejo' in obj: target_ids = [51, 52] 
    elif 'techo' in obj or 'ceiling' in obj: target_ids = [56]
    elif 'ventana' in obj or 'window' in obj: target_ids = [58]
    
    # Salón / Dormitorio / Comedor
    elif 'silla' in obj: target_ids = [62]
    elif 'sofa' in obj or 'sillón' in obj: target_ids = [63]
    elif 'cama' in obj: target_ids = [65]
    elif 'mesa' in obj or 'comedor' in obj: target_ids = [67]
    elif 'tv' in obj or 'televisor' in obj: target_ids = [72]
    
    # Cocina / Baño
    elif 'inodoro' in obj or 'wc' in obj or 'váter' in obj or 'toilet' in obj: target_ids = [70]
    elif 'lavabo' in obj or 'fregadero' in obj or 'sink' in obj: target_ids = [81]
    elif 'horno' in obj or 'oven' in obj: target_ids = [79]
    elif 'nevera' in obj or 'frigorífico' in obj or 'refrigerator' in obj: target_ids = [82]
    
    # Iluminación
    elif 'luz' in obj or 'lámpara' in obj: target_ids = [86]
    
    encontrado = False
    for segment in info:
        if segment['label_id'] in target_ids:
            mascara[mapa == segment['id']] = 255
            encontrado = True
            
    if not encontrado:
        print(f"⚠️ IA no reconoció el contorno exacto de '{nombre_objeto}'. Usando zona central de seguridad.")
        # Solución del fallo <i8: creamos una máscara limpia en formato "L" (escala de grises)
        w, h = imagen.size
        mascara_img = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(mascara_img)
        draw.rectangle([w//4, h//4, 3*w//4, 3*h//4], fill=255)
        return mascara_img

    return Image.fromarray(mascara).convert("L")

def extraer_mlsd(imagen: Image.Image) -> Image.Image:
    img_gray = cv2.cvtColor(np.array(imagen), cv2.COLOR_RGB2GRAY)
    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(img_gray)
    drawn_img = np.zeros_like(img_gray)
    if lines is not None: lsd.drawSegments(drawn_img, lines)
    return Image.fromarray(drawn_img).convert("RGB")

# --- PIPELINE PRINCIPAL UNIVERSAL ---
def motor_deco_ai(ruta_o_url: str, peticion: str, medidas: str = "No aportadas") -> Tuple[Dict, Image.Image, Image.Image]:
    img_orig = preparar_imagen_pro(ruta_o_url)

    print("🧠 Gemini evaluando la arquitectura y materiales de la estancia...")
    prompt_gemini = f"""
    Eres un arquitecto y diseñador de interiores experto en proyectos integrales (cocinas, baños, salones, exteriores, etc).
    Cliente pide: '{peticion}'. Medidas aportadas: '{medidas}'.
    
    1. Identifica qué tipo de estancia es mirando la foto.
    2. Si pide cambiar encimeras, sanitarios, electrodomésticos, suelos o paredes, clasifícalo como 'parcial' e indica la palabra clave (ej: 'inodoro', 'nevera', 'suelo', 'pared') en objeto_a_reemplazar.
    3. Asigna las tiendas correctas: Leroy Merlin/Bauhaus para bricolaje, baños, cocinas y suelos. IKEA/Zara Home/Maisons du Monde/Kave Home para muebles y deco.
    """
    
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[img_orig, prompt_gemini],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AnalisisDecoracionPro,
        )
    )
    datos = json.loads(response.text)

    # Informamos por pantalla qué estancia ha detectado
    print(f"📍 Estancia detectada: {datos['tipo_estancia'].upper()}")

    # 2. MÁSCARA INTELIGENTE
    mask_img = generar_mascara(img_orig, datos['objeto_a_reemplazar'])
    fuerza = 0.95 if datos['modo_generacion'] == 'completo' else 0.75

    # 3. ESTRUCTURA Y GENERACIÓN
    print(f"🎨 Renderizando ({datos['modo_generacion']} - {datos['objeto_a_reemplazar']})...")
    mlsd_img = extraer_mlsd(img_orig)
    
    img_final = pipe(
        prompt=datos['prompt_generacion'] + ", photorealistic, architectural digest, 8k resolution, highly detailed",
        negative_prompt="outdoors, nature, landscape, cartoon, warped lines, messy, unrealistic lighting",
        image=img_orig,
        mask_image=mask_img,
        control_image=mlsd_img,
        num_inference_steps=30,
        controlnet_conditioning_scale=1.0, 
        strength=fuerza,
        guidance_scale=8.5
    ).images[0]

    return datos, make_image_grid([img_orig, img_final], rows=1, cols=2), mask_img

Generador de Links Multi-Tienda

In [ ]:
from IPython.display import display, HTML

def generar_tienda_interactiva(datos: Dict):
    print("🛒 Generando enlaces afiliados y carrito...")
    
    html = f"""
    <div style='background: #fafafa; padding: 20px; border-radius: 10px; font-family: sans-serif; max-width: 700px;'>
        <h2 style='color: #2c3e50; border-bottom: 2px solid #e0e0e0; padding-bottom: 10px;'>
            🧾 Tu Proyecto: {datos['presupuesto_total_estimado']}€
        </h2>
        <p style='color: #555; font-size: 14px; line-height: 1.5;'><i>{datos['analisis_diseno']}</i></p>
        <ul style='list-style: none; padding: 0;'>
    """

    for item in datos['lista_compra']:
        nombre = item['nombre']
        tienda = item['tienda_sugerida'].lower()
        query = urllib.parse.quote(nombre)
        
        # Enrutador inteligente de tiendas
        link = f"https://www.google.com/search?q=comprar+{query}+{tienda}" # Fallback
        
        if 'leroy' in tienda: link = f"https://www.leroymerlin.es/buscar?q={query}"
        elif 'ikea' in tienda: link = f"https://www.ikea.com/es/es/search/?q={query}"
        elif 'amazon' in tienda: link = f"https://www.amazon.es/s?k={query}"
        elif 'zara' in tienda: link = f"https://www.zarahome.com/es/search.html?keyword={query}"
        elif 'bauhaus' in tienda: link = f"https://www.bauhaus.es/search?text={query}"
        elif 'maisons' in tienda: link = f"https://www.maisonsdumonde.com/ES/es/q/{query}"
        elif 'kave' in tienda: link = f"https://kavehome.com/es/es/search/?q={query}"

        # Colores por categoría para visualización
        color_cat = "#3498db" if item['categoria'] == 'iluminacion' else "#e67e22" if item['categoria'] in ['suelo', 'pared'] else "#27ae60"

        html += f"""
        <li style='background: white; margin-top: 10px; padding: 15px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); display: flex; justify-content: space-between; align-items: center;'>
            <div>
                <span style='font-size: 16px; font-weight: bold; color: #333;'>{nombre.title()}</span>
                <span style='font-size: 11px; background: {color_cat}; color: white; padding: 2px 6px; border-radius: 10px; margin-left: 8px;'>{item['categoria'].upper()}</span>
                <br><span style='color: #777; font-size: 13px;'>📏 {item['detalles_medidas']} | 🏬 {item['tienda_sugerida']}</span>
            </div>
            <div style='text-align: right;'>
                <span style='font-size: 18px; font-weight: bold; color: #2c3e50; display: block;'>~{item['precio_estimado_eur']}€</span>
                <a href='{link}' target='_blank' style='display: inline-block; margin-top: 5px; background: #111; color: white; padding: 6px 12px; text-decoration: none; border-radius: 5px; font-size: 12px;'>Ver Producto</a>
            </div>
        </li>
        """

    html += "</ul></div>"
    return html

Ejecución y Pruebas

In [ ]:
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN DEL CLIENTE ---
mi_foto = "https://images.hola.com/imagenes/decoracion/20210729193739/como-limpiar-sofa-tela-metodos-caseros-am/0-979-913/limpiar-sofa-am-t.jpg"

# Prueba pedir un cambio estructural:
mi_peticion = "Quiero cambiar ÚNICAMENTE la cama por una cama de diseño moderno con cabecero tapizado gris. Mantén el resto de la habitación exactamente igual."
mis_medidas = "La habitación tiene 20 metros cuadrados"

try:
    # 1. Ejecutar el Motor
    datos_finales, grid_antes_despues, img_mascara = motor_deco_ai(mi_foto, mi_peticion, mis_medidas)

    # 2. Mostrar Resultados Gráficos
    plt.figure(figsize=(18, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img_mascara, cmap='gray')
    plt.title(f"ZONA AISLADA: {datos_finales['objeto_a_reemplazar'].upper()}")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(grid_antes_despues)
    plt.title("RESULTADO: Original vs Propuesta AI")
    plt.axis("off")
    plt.show()

    # 3. Mostrar Carrito Interactivo
    display(HTML(generar_tienda_interactiva(datos_finales)))

except Exception as e:
    print(f"❌ Error en el motor: {e}")